In [26]:
import fitz  # PyMuPDF: opens the PDF and pulls out its text and images
from langchain_core.documents import Document  # standard LangChain box that holds a piece of text plus its info
from transformers import CLIPProcessor, CLIPModel  # CLIP: turns both images and text into vectors that can be compared
from PIL import Image  # opens and handles image files (the pictures taken from the PDF)
import torch  # the engine that runs the CLIP model
import numpy as np  # helps work with the lists of numbers (vectors)
from langchain.chat_models import init_chat_model  # loads the LLM that will read the images and write the answer
from langchain_core.prompts import PromptTemplate  # reusable prompt text with blanks like {context} and {question}
from langchain_core.messages import HumanMessage  # builds a user message that can carry both text and images
from sklearn.metrics.pairwise import cosine_similarity  # measures how close two vectors are, to find the best match
import os  # works with files and folders
import base64  # turns an image into text so it can be sent to the LLM
import io  # keeps image data in memory, so no temporary files are needed
from langchain_text_splitters import RecursiveCharacterTextSplitter  # cuts long text into small chunks
from langchain_community.vectorstores import FAISS  # stores the vectors and searches them quickly


In [27]:
# loads the .env file that holds the API keys

from dotenv import load_dotenv  
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# loads the CLIP model and processor that will turn both text and images into vectors

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")# convert prepared data to vectors
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32") #for data preprocessing
clip_model.eval()  # sets the model to evaluation mode, so it doesn't try to learn anything new

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 10483.46it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [14]:
### Embedding functions
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")  # resize and normalize the image for CLIP
    with torch.no_grad():  # no learning here, so it runs faster and uses less memory
        features = clip_model.get_image_features(**inputs)  # image -> 512 numbers
        if not isinstance(features, torch.Tensor):  # newer transformers return a wrapper object
            features = features.pooler_output
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,  # CLIP's max token length
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)  # text -> 512 numbers
        if not isinstance(features, torch.Tensor):  # newer transformers return a wrapper object
            features = features.pooler_output
        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


In [ ]:
## opens the PDF and prepares to extract its text and images
pdf_path = "../Nova_Labs_Annual_Report_2025.pdf"
doc=fitz.open(pdf_path)  #open th doc
all_docs = []  
all_embeddings = []  
image_data_storage={} #informat base64

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)  


In [ ]:
#Process text
for i, page in enumerate(doc):  # go through the PDF page by page 
    ## process text
    text = page.get_text()  
    if text.strip():  # skip pages that have no text
        ## create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = text_splitter.split_documents([temp_doc])  # cut the page text into small chunks

        # Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)  # chunk -> 512 numbers
            all_embeddings.append(embedding)  # keep the vector
            all_docs.append(chunk)  # keep the chunk, so we can show it later




In [17]:
## process images
## Three important actions:
## 1. convert the PDF image to PIL format
## 2. store it as base64 (the LLM that reads images needs base64 images)
## 3. create a CLIP embedding for retrieval

for i, page in enumerate(doc):  # go through the PDF page by page again 
    for img_index, img in enumerate(page.get_images(full=True)):  # every image found on this page
        try:
            xref = img[0]  # the PDF's internal id of this image
            base_image = doc.extract_image(xref)  # pull the image out of the PDF
            image_bytes = base_image["image"]  # the raw bytes of the image

            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"  # for example page_0_img_0

            # Store image as base64 for later use with the LLM
            buffered = io.BytesIO()  # an empty in-memory file
            pil_image.save(buffered, format="PNG")  # save the image into it as PNG
            img_base64 = base64.b64encode(buffered.getvalue()).decode()  # bytes -> text
            image_data_storage[image_id] = img_base64  # keep it, so we can send it to the LLM later

            # Embed image using CLIP
            embedding = embed_image(pil_image)  # image -> 512 numbers
            all_embeddings.append(embedding)

            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",  # a short label, since an image has no text
                metadata={"page": i, "type": "image", "image_id": image_id},
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()  # we are done with the PDF, so close the doc


In [22]:
# turn the list of vectors into one NumPy table (rows = chunks/images, columns = 512 numbers)
embeddings_array = np.array(all_embeddings)

# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs],
)
vector_store


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [28]:
# Gemini reads its key from GOOGLE_API_KEY,
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

# loads the Gemini LLM that will read the retrieved text and images and write the answer
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
llm


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-google-genai': '4.4.2'}}, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x000001E3DC391FD0>, default_metadata=(), model_kwargs={})

In [29]:
def retrieve_multimodal(query, k=5, k_images=2):
    """Unified retrieval using CLIP embeddings for both text and images.
    Returns the k best text chunks and the k_images best images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)

    # look at every stored item before filtering (the store is small)
    fetch_k = vector_store.index.ntotal

    # Search the text chunks
    text_results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k,
        filter={"type": "text"},
        fetch_k=fetch_k,
    )

    # Search the images separately, because CLIP scores text-to-text
    # higher than text-to-image, so images would never win a shared search
    image_results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k_images,
        filter={"type": "image"},
        fetch_k=fetch_k,
    )

    return text_results + image_results


In [30]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for the LLM."""
    content = []  # the message is a list of parts: some text parts, some image parts

    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n",
    })

    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"  # label each excerpt with its page
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n",
        })

    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_storage:  # only if we stored this image
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n",
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_storage[image_id]}",  # the image as base64 text
                },
            })

    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images.",
    })

    return HumanMessage(content=content)


In [33]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)

    # Create multimodal message
    message = create_multimodal_message(query, context_docs)

    # Get response from the LLM (Gemini)
    response = llm.invoke([message])

    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")

    return response.content  # the answer text


In [34]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "Which quarter had the highest revenue, and why was the next quarter lower?",
        "What share of the revenue came from each product?",
        "How did the monthly active users change during 2025, and what caused the jump?",
        "Which region has the lowest satisfaction score, and what is the goal for it in 2026?",
        "Summarize the main findings from the report in three sentences.",
    ]

    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)



Query: Which quarter had the highest revenue, and why was the next quarter lower?
--------------------------------------------------

Retrieved 7 documents:
  - Text from page 0: shows user growth. The company ended the year with 42 employees and opened a second office
in Singap...
  - Text from page 2: 4. Customers and Regions
Nova Labs serves customers in 38 countries. Europe is the largest market by...
  - Text from page 0: Nova Labs Annual Report 2025
A fictional company report used as sample data for multimodal RAG pract...
  - Text from page 2: 9,400
21,000
Support tickets resolved within 24 hours
81%
92%
6. Outlook for 2026
In 2026, Nova Labs...
  - Text from page 1: 2. User Growth
User growth was slow in the first half of the year and accelerated after the Nova Age...
  - Image from page 0
  - Image from page 1


Answer: The third quarter (Q3) had the highest revenue at $3.9 million. The next quarter (Q4) was lower because several large customers moved their annual payments to